# Sentinel-2 Image Matching Demo

This notebook demonstrates keypoint detection and image matching between two Sentinel-2 images of the same geographical tile captured on different dates.

The pipeline includes:

1. SIFT keypoint detection and descriptor extraction;
2. FLANN-based descriptor matching;
3. Lowe's ratio test;
4. geometric verification with RANSAC;
5. visualization of detected keypoints and final matches.

In [58]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import rasterio

In [59]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Task2_Sentinel2_Matching")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"

image_paths = sorted(PROCESSED_DIR.glob("*.png"))

In [61]:
image_1_bgr = cv2.imread(str(image_paths[0]))
image_2_bgr = cv2.imread(str(image_paths[1]))

if image_1_bgr is None or image_2_bgr is None:
    raise ValueError("One or both images could not be loaded.")

image_1_rgb = cv2.cvtColor(image_1_bgr, cv2.COLOR_BGR2RGB)
image_2_rgb = cv2.cvtColor(image_2_bgr, cv2.COLOR_BGR2RGB)

image_1_gray = cv2.cvtColor(image_1_bgr, cv2.COLOR_BGR2GRAY)
image_2_gray = cv2.cvtColor(image_2_bgr, cv2.COLOR_BGR2GRAY)

print("Image 1 shape:", image_1_rgb.shape)
print("Image 2 shape:", image_2_rgb.shape)

Image 1 shape: (2000, 2000, 3)
Image 2 shape: (2000, 2000, 3)


In [62]:
plt.figure(figsize=(10, 10))
plt.imshow(image_1_rgb)
plt.title(image_paths[0].name)
plt.axis("off")
plt.show()

plt.figure(figsize=(10, 10))
plt.imshow(image_2_rgb)
plt.title(image_paths[1].name)
plt.axis("off")
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## SIFT keypoint detection

SIFT detects distinctive local structures that are relatively robust to changes in scale, rotation and illumination.

For satellite images, the number of detected keypoints can be very large. Therefore, the detector is limited to the strongest 5000 features.

In [63]:
sift = cv2.SIFT_create(
    nfeatures=5000,
    contrastThreshold=0.04,
    edgeThreshold=10,
    sigma=1.6
)

keypoints_1, descriptors_1 = sift.detectAndCompute(
    image_1_gray,
    None
)

keypoints_2, descriptors_2 = sift.detectAndCompute(
    image_2_gray,
    None
)

print("Keypoints in image 1:", len(keypoints_1))
print("Keypoints in image 2:", len(keypoints_2))

if descriptors_1 is None or descriptors_2 is None:
    raise ValueError("SIFT descriptors could not be computed.")

Keypoints in image 1: 2658
Keypoints in image 2: 5000


In [64]:
keypoints_image_1 = cv2.drawKeypoints(
    image_1_rgb,
    keypoints_1,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

keypoints_image_2 = cv2.drawKeypoints(
    image_2_rgb,
    keypoints_2,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

plt.figure(figsize=(12, 12))
plt.imshow(keypoints_image_1)
plt.title(f"Detected SIFT keypoints: {len(keypoints_1)}")
plt.axis("off")
plt.show()

plt.figure(figsize=(12, 12))
plt.imshow(keypoints_image_2)
plt.title(f"Detected SIFT keypoints: {len(keypoints_2)}")
plt.axis("off")
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## Descriptor matching

The SIFT descriptors are matched using FLANN.

For each descriptor, the two nearest matches are retrieved. Lowe's ratio test is then used to reject ambiguous matches.

In [65]:
index_params = {
    "algorithm": 1,
    "trees": 5
}

search_params = {
    "checks": 100
}

flann = cv2.FlannBasedMatcher(
    index_params,
    search_params
)

knn_matches = flann.knnMatch(
    descriptors_1,
    descriptors_2,
    k=2
)

ratio_threshold = 0.75
good_matches = []

for match_pair in knn_matches:
    if len(match_pair) < 2:
        continue

    first, second = match_pair

    if first.distance < ratio_threshold * second.distance:
        good_matches.append(first)

print("Raw KNN matches:", len(knn_matches))
print("Matches after Lowe ratio test:", len(good_matches))

Raw KNN matches: 2658
Matches after Lowe ratio test: 110


In [66]:
matches_after_ratio = cv2.drawMatches(
    image_1_rgb,
    keypoints_1,
    image_2_rgb,
    keypoints_2,
    good_matches[:150],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(20, 10))
plt.imshow(matches_after_ratio)
plt.title(
    f"Matches after Lowe ratio test: "
    f"{len(good_matches)} total, showing up to 150"
)
plt.axis("off")
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## Geometric verification with RANSAC

Descriptor similarity alone can produce incorrect matches.

RANSAC estimates a homography between the two images and identifies geometrically consistent matches. These matches are treated as inliers.

In [67]:
if len(good_matches) < 4:
    raise ValueError(
        "At least 4 good matches are required to estimate a homography."
    )

In [68]:
source_points = np.float32(
    [
        keypoints_1[match.queryIdx].pt
        for match in good_matches
    ]
).reshape(-1, 1, 2)

destination_points = np.float32(
    [
        keypoints_2[match.trainIdx].pt
        for match in good_matches
    ]
).reshape(-1, 1, 2)

homography, inlier_mask = cv2.findHomography(
    source_points,
    destination_points,
    method=cv2.RANSAC,
    ransacReprojThreshold=5.0
)

if homography is None or inlier_mask is None:
    raise ValueError("Homography estimation failed.")

inlier_mask = inlier_mask.ravel().astype(bool)

inlier_matches = [
    match
    for match, is_inlier in zip(good_matches, inlier_mask)
    if is_inlier
]

outlier_matches = [
    match
    for match, is_inlier in zip(good_matches, inlier_mask)
    if not is_inlier
]

print("Good matches:", len(good_matches))
print("RANSAC inliers:", len(inlier_matches))
print("RANSAC outliers:", len(outlier_matches))
print(
    "Inlier ratio:",
    f"{len(inlier_matches) / len(good_matches):.2%}"
)

Good matches: 110
RANSAC inliers: 89
RANSAC outliers: 21
Inlier ratio: 80.91%


In [69]:
final_matches_image = cv2.drawMatches(
    image_1_rgb,
    keypoints_1,
    image_2_rgb,
    keypoints_2,
    inlier_matches[:150],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(20, 10))
plt.imshow(final_matches_image)
plt.title(
    f"Geometrically verified matches: "
    f"{len(inlier_matches)} inliers"
)
plt.axis("off")
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [70]:
output_path = OUTPUT_DIR / "final_verified_matches.png"

cv2.imwrite(
    str(output_path),
    cv2.cvtColor(final_matches_image, cv2.COLOR_RGB2BGR)
)

print("Saved:", output_path)

Saved: /content/drive/MyDrive/Task2_Sentinel2_Matching/outputs/final_verified_matches.png


In [71]:
results = {
    "image_1_keypoints": len(keypoints_1),
    "image_2_keypoints": len(keypoints_2),
    "raw_knn_matches": len(knn_matches),
    "matches_after_ratio_test": len(good_matches),
    "ransac_inliers": len(inlier_matches),
    "ransac_outliers": len(outlier_matches),
    "inlier_ratio": len(inlier_matches) / len(good_matches)
}

for metric, value in results.items():
    if metric == "inlier_ratio":
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")

image_1_keypoints: 2658
image_2_keypoints: 5000
raw_knn_matches: 2658
matches_after_ratio_test: 110
ransac_inliers: 89
ransac_outliers: 21
inlier_ratio: 80.91%


In [72]:
ratio_output_path = OUTPUT_DIR / "matches_after_ratio.png"

cv2.imwrite(
    str(ratio_output_path),
    cv2.cvtColor(matches_after_ratio, cv2.COLOR_RGB2BGR)
)

print(f"Saved: {ratio_output_path}")

Saved: /content/drive/MyDrive/Task2_Sentinel2_Matching/outputs/matches_after_ratio.png


## Conclusion

The SIFT–FLANN pipeline successfully detects and matches local features between two Sentinel-2 images of the same geographical area.

Lowe's ratio test removes ambiguous descriptor matches, while RANSAC rejects geometrically inconsistent correspondences.

The final inlier matches represent feature correspondences that are both visually similar and spatially consistent.

Potential improvements include:

- using image patches instead of resizing the full Sentinel-2 tile;
- cloud masking before keypoint detection;
- testing additional spectral bands;
- comparing SIFT with learned methods such as SuperPoint, SuperGlue or LoFTR;
- evaluating the algorithm on more locations and seasons.